In [11]:
from libcellml import Parser, Validator, Importer
from pathlib import Path

In [39]:
def print_math(model):
    for i in range(model.componentCount()):
        comp = model.component(i)

        math = comp.math()
        if math:
            print(f"\nMath in component '{comp.name()}':")
            print(math)

In [40]:
def print_components(model):
    print(f"Model: {model.name()}")
    print(f"Number of components: {model.componentCount()}")

    for i in range(model.componentCount()):
        comp = model.component(i)
        print(f"\nComponent {i}: {comp.name()}")

In [27]:
def parse_cellml_file(path: str):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    print(text[:10])
    parser = Parser()
    parser.setStrict(False)
    model = parser.parseModel(text)

    issues = []
    for i in range(parser.errorCount()):
        issues.append(parser.error(i).description())

    return model, issues

In [46]:
def print_variables(model):
    total = 0
    for i in range(model.componentCount()):
        comp = model.component(i)
        print(f"\nComponent: {comp.name()}")

        for j in range(comp.variableCount()):
            var = comp.variable(j)

            print(f"  Variable: {var.name()}")
            print(f"    Units: {var.units().name() if var.units() else None}")
            print(f"    Initial value: {var.initialValue()}")
            print(f"    Interface (public): {var.interfaceType()}")
            total += 1

    print(total)

In [6]:
def flatten_model(importer, model):
    flat_model = importer.flattenModel(model)
    return flat_model

In [28]:
model, issues = parse_cellml_file("albrecht_colegrove_friel_2002.cellml")
print(*issues, sep=", ")

<?xml vers



In [42]:
print(model.componentCount())
print_components(model)

7
Model: albrecht_colegrove_friel_2002_version01
Number of components: 7

Component 0: environment

Component 1: cytoplasmic_calcium

Component 2: intraluminal_calcium

Component 3: total_cytoplasmic_Ca_flux

Component 4: Ca_extrusion_across_the_plasma_membrane

Component 5: Ca_uptake_by_SR_Ca_ATPase

Component 6: ER_Ca_release


In [47]:
print_variables(model)


Component: environment
  Variable: time
    Units: second
    Initial value: 
    Interface (public): public

Component: cytoplasmic_calcium
  Variable: Ca_i
    Units: nanomolar
    Initial value: 
    Interface (public): public
  Variable: J_i
    Units: nanomolar_per_second
    Initial value: 
    Interface (public): 
  Variable: J_ER
    Units: nanomolar_per_second
    Initial value: 
    Interface (public): public
  Variable: J_pm
    Units: nanomolar_per_second
    Initial value: 
    Interface (public): public
  Variable: time
    Units: second
    Initial value: 
    Interface (public): public

Component: intraluminal_calcium
  Variable: Ca_ER
    Units: nanomolar
    Initial value: 
    Interface (public): public
  Variable: v_ER
    Units: micro_litre
    Initial value: 
    Interface (public): 
  Variable: k_ER
    Units: dimensionless
    Initial value: 
    Interface (public): 
  Variable: Ca_ER_init
    Units: nanomolar
    Initial value: 
    Interface (public): 
  Vari

In [48]:
def print_connections(model):
    total = 0
    for i in range(model.componentCount()):
        comp = model.component(i)

        for j in range(comp.variableCount()):
            var = comp.variable(j)

            for k in range(var.equivalentVariableCount()):
                eq_var = var.equivalentVariable(k)

                print(f"{comp.name()}.{var.name()} <-> "
                      f"{eq_var.parent().name()}.{eq_var.name()}")
                total += 1

    print (total)

In [49]:
print_connections(model)

environment.time <-> cytoplasmic_calcium.time
environment.time <-> intraluminal_calcium.time
environment.time <-> total_cytoplasmic_Ca_flux.time
environment.time <-> Ca_extrusion_across_the_plasma_membrane.time
environment.time <-> Ca_uptake_by_SR_Ca_ATPase.time
environment.time <-> ER_Ca_release.time
cytoplasmic_calcium.Ca_i <-> Ca_extrusion_across_the_plasma_membrane.Ca_i
cytoplasmic_calcium.Ca_i <-> Ca_uptake_by_SR_Ca_ATPase.Ca_i
cytoplasmic_calcium.Ca_i <-> ER_Ca_release.Ca_i
cytoplasmic_calcium.Ca_i <-> intraluminal_calcium.Ca_i
cytoplasmic_calcium.J_ER <-> ER_Ca_release.J_ER
cytoplasmic_calcium.J_pm <-> Ca_extrusion_across_the_plasma_membrane.J_pm
cytoplasmic_calcium.time <-> environment.time
intraluminal_calcium.Ca_ER <-> ER_Ca_release.Ca_ER
intraluminal_calcium.Ca_i <-> cytoplasmic_calcium.Ca_i
intraluminal_calcium.v_i <-> total_cytoplasmic_Ca_flux.v_i
intraluminal_calcium.k_i <-> total_cytoplasmic_Ca_flux.k_i
intraluminal_calcium.J_ER <-> ER_Ca_release.J_ER
intraluminal_calciu

In [5]:
def validate_model(model):
    validator = Validator()
    validator.validateModel(model)

    issues = []
    for i in range(validator.errorCount()):
        issues.append(validator.error(i).description())

    return issues

In [30]:
validate_model(model)

[]

In [32]:
importer = Importer()
flatten_model(importer, model)

<libcellml.model.Model; proxy of <Swig Object of type 'std::shared_ptr< libcellml::Model > *' at 0x11069f750> >

In [35]:
component1 = model.component(1)
component1

<libcellml.component.Component; proxy of <Swig Object of type 'std::shared_ptr< libcellml::Component > *' at 0x110d487b0> >

In [38]:
print(component1.variableCount())
for i in range(component1.variableCount()):
    print(component1.variable(i).name())

5
Ca_i
J_i
J_ER
J_pm
time


In [7]:
from libcellml import Analyser

def analyse_model(flat_model):
    analyser = Analyser()
    analyser.analyseModel(flat_model)

    issues = []
    for i in range(analyser.errorCount()):
        issues.append(analyser.error(i).description())

    return analyser, issues

In [8]:
def inspect_flat_model(flat_model):
    data = []

    for i in range(flat_model.componentCount()):
        comp = flat_model.component(i)
        comp_info = {
            "component": comp.name(),
            "variables": []
        }

        for j in range(comp.variableCount()):
            var = comp.variable(j)
            comp_info["variables"].append({
                "name": var.name(),
                "units": var.units().name() if var.units() is not None else None,
                "initial_value": var.initialValue(),
            })

        comp_info["math"] = comp.math()
        data.append(comp_info)

    return data

In [21]:

def collect_issues(source):
    issues = []
    if hasattr(source, "errorCount"):
        for i in range(source.errorCount()):
            issues.append(source.error(i).description())
    return issues

def run_libcellml_pipeline(cellml_path: str):
    cellml_path = Path(cellml_path)

    # Parse
    parser = Parser()
    text = cellml_path.read_text(encoding="utf-8")
    model = parser.parseModel(text)
    parse_issues = collect_issues(parser)

    # Validate
    validator = Validator()
    validator.validateModel(model)
    validation_issues = collect_issues(validator)

    # Resolve imports
    importer = Importer()
    importer.resolveImports(model, str(cellml_path.parent))
    import_issues = collect_issues(importer)

    # Flatten
    flat_model = importer.flattenModel(model)

    # Analyse
    analyser = Analyser()
    analyser.analyseModel(flat_model)
    analysis_issues = collect_issues(analyser)

    return {
        "model": model,
        "flat_model": flat_model,
        "parse_issues": parse_issues,
        "validation_issues": validation_issues,
        "import_issues": import_issues,
        "analysis_issues": analysis_issues,
        "analyser": analyser,
    }

In [22]:
model = run_libcellml_pipeline("albrecht_colegrove_friel_2002.cellml")['model']

0